In [ ]:
"""
Run only if scikit-learn isn't already installed in your environment. You can also run this in a terminal, not just a notebook.
"""

# pip install scikit-learn numpy matplotlib plotly

In [ ]:
"""
Run only if nbformat isn't already installed in your environment. You can also run this in a terminal, not just a notebook.
"""

import sys 
!{sys.executable} -m pip install --upgrade "nbformat>=4.2.0"

In [5]:
import numpy as np
from sklearn.datasets import fetch_openml

# Load MNIST dataset (70,000 images of 28x28)
print("Downloading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X_full, y_full = mnist.data, mnist.target.astype(int)

# Number of classes and samples per class
n_classes = 10
samples_per_class = 100  # 100 per digit for total of 1000

X_list, y_list = [], []

# Collect 100 samples per class
for label in range(n_classes):
    idx = np.where(y_full == label)[0]
    if samples_per_class > idx.size:
        # fail early with a clear message instead of letting numpy blow up
        raise ValueError(
            f"requested {samples_per_class} samples for class {label} "
            f"but only {idx.size} examples are available; "
            "reduce samples_per_class or set replace=True"
        )
    chosen = np.random.choice(idx, samples_per_class, replace=False)
    X_list.append(X_full[chosen])
    y_list.append(y_full[chosen])

# Stack to get final balanced dataset
X = np.vstack(X_list)
y = np.hstack(y_list)

# Shuffle the dataset
perm = np.random.permutation(len(y))
X, y = X[perm], y[perm]

print("Balanced subset created:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Class distribution:", np.bincount(y))


Balanced subset created:
X shape: (1000, 784)
y shape: (1000,)
Class distribution: [100 100 100 100 100 100 100 100 100 100]


In [9]:
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- Assumes you already have X and y from the MNIST subset step ---

# 1) Standardize features
scaler = StandardScaler(with_mean=True, with_std=True)
X_std = scaler.fit_transform(X)

# 2) Run PCA
pca = PCA(n_components=min(X_std.shape), random_state=0)
X_pca = pca.fit_transform(X_std)

# 3) Explained variance ratios
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)
components = np.arange(1, len(evr) + 1)

# 4) Create interactive scree plot
fig = go.Figure()

# Bar trace for individual explained variance
fig.add_trace(go.Bar(
    x=components,
    y=evr,
    name='Explained Variance Ratio',
    marker=dict(color='rgba(31, 119, 180, 0.7)')
))

# Line trace for cumulative variance
fig.add_trace(go.Scatter(
    x=components,
    y=cum_evr,
    mode='lines+markers',
    name='Cumulative Explained Variance',
    line=dict(color='firebrick', width=3)
))

# 5) Figure layout
fig.update_layout(
    title='Scree Plot — PCA on MNIST (Standardized)',
    xaxis_title='Principal Component',
    yaxis_title='Proportion of Variance Explained',
    legend=dict(x=0.7, y=0.1),
    template='plotly_white'
)

fig.show()

# 6) (Optional) Print components needed for 95% variance
threshold = 0.95
k_95 = np.argmax(cum_evr >= threshold) + 1
print(f"Number of components to reach {threshold*100:.0f}% variance: {k_95}")

# Reduced dataset (optional)
X_reduced = X_pca[:, :k_95]
print("Reduced X shape:", X_reduced.shape)


Number of components to reach 95% variance: 189
Reduced X shape: (1000, 189)


In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def pca_by_variance(X, retain=0.95, random_state=0, return_model=True):
    """
    Run PCA keeping enough components to reach a target variance threshold.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
        Input features.
    retain : float or int
        If in (0, 1], interpreted as proportion (e.g., 0.95).
        If in (1, 100], interpreted as percentage (e.g., 95).
    random_state : int
        Random seed for reproducibility where applicable.
    return_model : bool
        If True, also return the fitted scaler and PCA model.

    Returns
    -------
    X_reduced : np.ndarray, shape (n_samples, k)
        PCA-transformed data with k components.
    info : dict
        {
          'k': number of components kept,
          'explained_variance_ratio': per-component EVR,
          'cumulative_explained_variance': cumulative EVR,
        }
    (optional) scaler : StandardScaler
    (optional) pca : PCA
    """
    # normalise the argument
    if isinstance(retain, str):
        tmp = retain.strip().rstrip('%')
        try:
            retain = float(tmp)
        except ValueError:
            raise ValueError(f"cannot interpret retain={retain!r}") from None
        
    # Normalize retain to [0,1]
    if retain > 1:
        retain = float(retain) / 100.0
    if not (0 < retain <= 1):
        raise ValueError("`retain` must be in (0,1] or (1,100].")

    # 1) Standardize
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_std = scaler.fit_transform(X)

    # 2) PCA with target variance
    # sklearn will choose the minimal number of PCs whose cumulative EVR >= retain
    pca = PCA(n_components=retain, random_state=random_state)
    X_reduced = pca.fit_transform(X_std)

    # 3) Collect info
    evr = pca.explained_variance_ratio_
    cum_evr = np.cumsum(evr)
    k = X_reduced.shape[1]

    info = {
        'k': k,
        'explained_variance_ratio': evr,
        'cumulative_explained_variance': cum_evr
    }

    if return_model:
        return X_reduced, info, scaler, pca
    return X_reduced, info


# ---------------------------
# Example usage
# ---------------------------
# Assuming X, y already exist in memory from your MNIST subset.
if 'X' in globals():
    # Keep 95% variance (equivalently, retain=95 also works)
    X_reduced, info, scaler, pca = pca_by_variance(X, retain=0.95)

    print(f"Target variance retained: 95%")
    print(f"Chosen number of components (k): {info['k']}")
    print(f"Cumulative EVR at k: {info['cumulative_explained_variance'][-1]:.4f}")
    print("Reduced X shape:", X_reduced.shape)
else:
    print("variables X and y are not defined – run the MNIST subset cell first")

Target variance retained: 95%
Chosen number of components (k): 189
Cumulative EVR at k: 0.9503
Reduced X shape: (1000, 189)


In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_pairwise_pcs_two_classes(X, y, class_pairs=None, marker_size=5, opacity=0.7):
    """
    For each pair of classes, show a 1x3 grid:
      (PC1 vs PC2), (PC1 vs PC3), (PC2 vs PC3)
    Only the two classes in the pair are plotted in each grid.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
    y : array-like, shape (n_samples,)
    class_pairs : list of tuple[int, int]
        Pairs of class labels to plot. If None, uses a few defaults.
    marker_size : int
        Marker size for scatter points.
    opacity : float
        Marker opacity (0–1).
    """
    if class_pairs is None:
        # A few illustrative pairs; customize as you like
        class_pairs = [(0, 1), (3, 8), (4, 9)]

    # 1) Standardize and PCA (top 3 PCs)
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_std = scaler.fit_transform(X)

    pca = PCA(n_components=3, random_state=0)
    Z = pca.fit_transform(X_std)  # shape: (n_samples, 3)

    pc_pairs = [("PC1", "PC2", 0, 1),
                ("PC1", "PC3", 0, 2),
                ("PC2", "PC3", 1, 2)]

    # 2) For each pair of classes, build a 1x3 grid
    for (c1, c2) in class_pairs:
        mask = (y == c1) | (y == c2)
        Z_sub = Z[mask]
        y_sub = y[mask]

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=[f"{xlab} vs {ylab}" for (xlab, ylab, _, _) in pc_pairs]
        )

        # Colors for the two classes in this plot
        class_colors = {c1: "royalblue", c2: "firebrick"}

        for col, (xlab, ylab, xi, yi) in enumerate(pc_pairs, start=1):
            for cls in (c1, c2):
                m = (y_sub == cls)
                fig.add_trace(
                    go.Scattergl(
                        x=Z_sub[m, xi],
                        y=Z_sub[m, yi],
                        mode="markers",
                        name=f"Class {cls}",
                        legendgroup=f"class_{cls}",
                        marker=dict(size=marker_size, opacity=opacity, color=class_colors[cls]),
                        showlegend=(col == 1)  # show legend only once (on first subplot)
                    ),
                    row=1, col=col
                )
            fig.update_xaxes(title_text=xlab, row=1, col=col)
            fig.update_yaxes(title_text=ylab, row=1, col=col)

        fig.update_layout(
            title=f"Pairwise PC Scatterplots — Classes {c1} vs {c2}",
            template="plotly_white",
            height=400,
            width=1100,
            margin=dict(l=40, r=20, t=60, b=40)
        )
        fig.show()


# ---------------------------
# Example usage (assumes X, y are in memory)
# ---------------------------
# Choose your class pairs explicitly, or let the function use defaults:
# class_pairs = [(0,1), (2,3), (4,5), (6,7), (8,9)]
class_pairs = [(0,1), (3,8), (4,9), (2,9)]

plot_pairwise_pcs_two_classes(X, y, class_pairs=class_pairs, marker_size=5, opacity=0.8)
